In [0]:
%run ./01-config

In [0]:
# Import PySpark SQL functions (e.g., aggregation, watermarking, joins)
from pyspark.sql import functions as F

# Construct absolute paths for raw data and streaming checkpoints
# - `landing_zone`: Directory where raw source files are staged (used in Bronze layer; referenced here for consistency)
# - `checkpoint_base`: Directory to store streaming query state and offsets for fault tolerance
landing_zone = base_dir_data + "/raw"
checkpoint_base = base_dir_checkpoint + "/checkpoints"


# --------------------------------------------------------------------------------------------------
# GENERIC UPSERT HELPER FOR DELTA MERGE (REUSED FROM SILVER LAYER)
# --------------------------------------------------------------------------------------------------

def upserter(df_micro_batch, batch_id, merge_query, temp_view):
    """
    Executes a SQL MERGE statement on a micro-batch DataFrame.
    
    - Creates a temporary view so the DataFrame can be referenced in the SQL query.
    - Runs the provided `merge_query` using the underlying SparkSession.
    - Prints a completion message for observability.
    
    This pattern enables idempotent upserts into Delta tables using SQL MERGE syntax.
    """
    df_micro_batch.createOrReplaceTempView(temp_view)
    df_micro_batch._jdf.sparkSession().sql(merge_query)
    print(f"Batch {batch_id} for {temp_view} processed.")


# --------------------------------------------------------------------------------------------------
# GOLD LAYER UPSERT FUNCTIONS (Silver → Gold Aggregation)
# --------------------------------------------------------------------------------------------------

def upsert_workout_bpm_summary(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates the Gold table `workout_bpm_summary` by aggregating heart rate data per workout session
    and enriching it with user demographics from the Silver `user_bins` table.
    
    - Reads from the Silver `workout_bpm` table as a change stream (Delta CDC).
    - Groups by session-level keys and computes min/avg/max heart rate + count.
    - Joins with static `user_bins` to add age, gender, city, and state for segmentation.
    - Insert-only (no updates expected for completed sessions).
    """
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.workout_bpm_summary a
    USING workout_bpm_summary_delta b
    ON a.user_id=b.user_id AND a.workout_id = b.workout_id AND a.session_id=b.session_id
    WHEN NOT MATCHED THEN INSERT *
    """

    # Load static demographic bins (Silver table) for enrichment
    df_users = spark.read.table(f"{catalog}.{db_name}.user_bins")

    # Stream from Silver `workout_bpm`, aggregate, and enrich
    df_delta = (spark.readStream
                    .option("startingVersion", startingVersion)  # Start from specific Delta version
                    .table(f"{catalog}.{db_name}.workout_bpm")
                    .withWatermark("end_time", "30 seconds")     # Enable state cleanup for aggregations
                    .groupBy("user_id", "workout_id", "session_id", "end_time")
                    .agg(
                        F.min("heartrate").alias("min_bpm"),
                        F.mean("heartrate").alias("avg_bpm"),
                        F.max("heartrate").alias("max_bpm"),
                        F.count("heartrate").alias("num_recordings")
                    )
                    .join(df_users, ["user_id"])  # Enrich with demographic bins
                    .select(
                        "workout_id", "session_id", "user_id", 
                        "age", "gender", "city", "state", 
                        "min_bpm", "avg_bpm", "max_bpm", "num_recordings"
                    )
                )

    # Write aggregated results to Gold table using foreachBatch + MERGE
    stream_writer = (df_delta.writeStream
                    .foreachBatch(lambda df, id: upserter(df, id, merge_query, "workout_bpm_summary_delta"))
                    .outputMode("append")  # Aggregations are final; no updates
                    .option("checkpointLocation", f"{checkpoint_base}/workout_bpm_summary")
                    .queryName("workout_bpm_summary_upsert_stream")
            )
    
    # Trigger as one-time batch or continuous stream
    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()
    


# --------------------------------------------------------------------------------------------------
# GOLD LAYER ORCHESTRATION
# --------------------------------------------------------------------------------------------------

def upsert_gold(once=True, processing_time="5 seconds"):
    """
    Orchestrates the Gold layer upsert process.
    
    Currently only includes `workout_bpm_summary`, but structured to support additional Gold tables.
    If `once=True`, waits for the streaming query to complete before returning.
    """
    import time
    start = int(time.time())
    print(f"\nExecuting gold layer upsert ...")
    upsert_workout_bpm_summary(once, processing_time)
    if once:
        # Block until the triggered batch query finishes
        for stream in spark.streams.active:
            stream.awaitTermination()
    print(f"Completed gold layer upsert {int(time.time()) - start} seconds") 


# --------------------------------------------------------------------------------------------------
# VALIDATION FUNCTIONS FOR GOLD LAYER
# --------------------------------------------------------------------------------------------------

def assert_count(catalog, db_name, table_name, expected_count, filter="true"):
    """
    Validates that a table contains exactly `expected_count` records matching an optional SQL filter.
    
    Used for tables where only row count matters (e.g., aggregated summaries).
    """
    print(f"Validating record counts in {table_name}...", end='')
    actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where(filter).count()
    assert actual_count == expected_count, (
        f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where {filter}"
    )
    print(f"Found {actual_count:,} / Expected {expected_count:,} records where {filter}: Success")


def assert_rows(catalog, db_name, test_data_dir, location, table_name, sets):
    """
    Performs **exact row-by-row validation** by comparing the actual table contents
    against a pre-generated expected Parquet file.
    
    - Expected data is stored as `{test_data_dir}/{location}_{sets}.parquet`
      (e.g., `7-gym_summary_1.parquet` for first test set).
    - Collects all rows from both actual and expected datasets and compares them directly.
    - Sensitive to row order and exact values—used for deterministic Gold views like `gym_summary`.
    """
    print(f"Validating records in {table_name}...", end='')
    expected_rows = spark.read.format("parquet").load(f"{test_data_dir}/{location}_{sets}.parquet").collect()
    actual_rows = spark.read.table(f"{catalog}.{db_name}.{table_name}").collect()
    assert expected_rows == actual_rows, (
        f"\n Data mismatch in {table_name}\n"
        f"- Expected: {len(expected_rows)} rows\n"
        f"- Actual:   {len(actual_rows)} rows"
    )
    print(f"Expected data matches with the actual data in {table_name}: Success")


def validate_gold(catalog, db_name, test_data_dir, sets):
    """
    Validates the Gold layer after upsert.
    
    - Uses **exact row comparison** for the `gym_summary` view (which is deterministic).
    - Uses **row count validation** for `workout_bpm_summary` only when `sets > 1`
      (likely because the first set produces 1 row, and the test expects 2 after two runs).
      
    This mixed approach reflects different validation needs:
      - Views (`gym_summary`) → full content match.
      - Aggregated tables (`workout_bpm_summary`) → count-based check in multi-set scenarios.
    """
    import time
    start = int(time.time())
    print(f"\nValidating gold layer records...")

    # Validate `gym_summary` by comparing against expected Parquet file
    assert_rows(catalog, db_name, test_data_dir, "7-gym_summary", "gym_summary", sets)

    # Validate `workout_bpm_summary` count only in multi-set scenarios
    if sets > 1:
        assert_count(catalog, db_name, "workout_bpm_summary", 2)

    print(f"Gold layer validation completed in {int(time.time()) - start} seconds")